# Reach Zarr point-cloud analyzer (Plotly)

Loads a pg3d reach Zarr dataset and renders the **final point cloud of a chosen episode**
as an interactive 3D Plotly scatter, color-coded by point type:

- **scene** points (world / table / object),
- **robot** points (`robot_mask`),
- **goal-marker** points (the trailing `goal_marker_points` slots — the oriented triad or legacy sphere),
- the **TCP** and **target position** anchors,
- a **TCP coordinate-frame triad** (x=red, y=green, z=blue; z is the approach axis) so you can see *where the end-effector lands and how it is oriented*,
- a **goal-pose triad** at the target (from `goal_quat`) to compare commanded vs achieved orientation.

The goal-marker tail count and style are read from the dataset's `metadata.json`
(`point_cloud_saliency`). No environment changes; requires only `numpy`, `zarr`, and `plotly`.

In [14]:
import json
from pathlib import Path

import numpy as np
import zarr

try:
    import plotly.graph_objects as go
except ImportError as exc:  # pragma: no cover
    raise SystemExit(
        "plotly is not installed in this environment. Install it in your kernel, e.g.\n"
        "    %pip install plotly\n"
        "(this notebook intentionally does not modify the project environment)."
    ) from exc

# ---- EDIT ME ----
DATASET = Path("/home/nitin/RRC/PG3D/artifacts/pose-reach-smoke/xarm7-nocalib.zarr")
EPISODE = 0            # 0-indexed episode to inspect
FRAME = "last_valid"   # "last_valid", "last", or an integer step within the episode
# -----------------

root = zarr.open_group(str(DATASET), mode="r")
data = root["data"]
episode_ends = np.asarray(root["meta"]["episode_ends"][:], dtype=np.int64)
num_episodes = int(len(episode_ends))
print(f"dataset: {DATASET}")
print(f"episodes: {num_episodes}, total steps: {int(episode_ends[-1]) if num_episodes else 0}")
print(f"arrays: {sorted(data.keys())}")

dataset: /home/nitin/RRC/PG3D/artifacts/pose-reach-smoke/xarm7-nocalib.zarr
episodes: 3, total steps: 210
arrays: ['action', 'eef_pos', 'goal_pos', 'goal_quat', 'goal_relative', 'point_cloud', 'point_valid_mask', 'robot_mask', 'sim_action', 'state', 'success', 'target_position', 'tcp_pose']


In [16]:
# Read goal-marker config from metadata.json (falls back to sane defaults).
meta_path = DATASET / "metadata.json"
goal_marker_points = 0
goal_marker_style = "unknown"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    saliency = meta.get("point_cloud_saliency", {})
    goal_marker_points = int(saliency.get("goal_marker_points", 0))
    goal_marker_style = str(saliency.get("goal_marker_style", "unknown"))
print(f"goal_marker_points = {goal_marker_points}, goal_marker_style = {goal_marker_style!r}")

goal_marker_points = 192, goal_marker_style = 'triad'


In [17]:
def episode_bounds(idx: int) -> tuple[int, int]:
    """Return [start, end) row indices for a 0-indexed episode."""
    if idx < 0 or idx >= num_episodes:
        raise IndexError(f"episode {idx} out of range [0, {num_episodes})")
    start = 0 if idx == 0 else int(episode_ends[idx - 1])
    end = int(episode_ends[idx])
    return start, end


def pick_frame(start: int, end: int, valid_mask_ep: np.ndarray) -> int:
    """Resolve the FRAME selector to an absolute row index within [start, end)."""
    n = end - start
    if isinstance(FRAME, int):
        rel = FRAME if FRAME >= 0 else n + FRAME
        rel = int(np.clip(rel, 0, n - 1))
        return start + rel
    if FRAME == "last":
        return end - 1
    if FRAME == "last_valid":
        # Last step that has any valid points; falls back to last step.
        any_valid = valid_mask_ep.any(axis=1)
        rel = int(np.flatnonzero(any_valid)[-1]) if any_valid.any() else n - 1
        return start + rel
    raise ValueError(f"unsupported FRAME={FRAME!r}")


start, end = episode_bounds(EPISODE)

pc_ep = np.asarray(data["point_cloud"][start:end], dtype=np.float32)          # [T, N, 3]
valid_ep = (
    np.asarray(data["point_valid_mask"][start:end], dtype=bool)
    if "point_valid_mask" in data else np.ones(pc_ep.shape[:2], dtype=bool)
)
robot_ep = (
    np.asarray(data["robot_mask"][start:end], dtype=bool)
    if "robot_mask" in data else np.zeros(pc_ep.shape[:2], dtype=bool)
)
target_ep = np.asarray(data["target_position"][start:end], dtype=np.float32)  # [T, 3]
tcp_ep = np.asarray(data["tcp_pose"][start:end], dtype=np.float32)            # [T, 7]
goal_quat_ep = (
    np.asarray(data["goal_quat"][start:end], dtype=np.float32)
    if "goal_quat" in data else None
)

row = pick_frame(start, end, valid_ep)
rel = row - start
print(f"episode {EPISODE}: steps [{start}, {end}) (len {end - start}); showing frame {row} (rel {rel})")

episode 0: steps [0, 72) (len 72); showing frame 71 (rel 71)


In [18]:
# Slice the chosen frame and split into scene / robot / goal-marker groups.
N = pc_ep.shape[1]
pts = pc_ep[rel]                 # [N, 3]
valid = valid_ep[rel]            # [N]
robot = robot_ep[rel]            # [N]

# Goal-marker slots are the trailing `goal_marker_points` rows of the cloud.
goal_slot = np.zeros(N, dtype=bool)
if 0 < goal_marker_points < N:
    goal_slot[-goal_marker_points:] = True

is_goal = goal_slot & valid
is_robot = robot & valid & ~goal_slot
is_scene = valid & ~robot & ~goal_slot

target_xyz = target_ep[rel]
tcp_xyz = tcp_ep[rel, :3]
print(
    f"points: scene={int(is_scene.sum())}, robot={int(is_robot.sum())}, "
    f"goal_marker={int(is_goal.sum())}, invalid={int((~valid).sum())}"
)
if goal_quat_ep is not None:
    print(f"goal_quat (w,x,y,z) = {np.round(goal_quat_ep[rel], 4).tolist()}")

points: scene=0, robot=518, goal_marker=192, invalid=314
goal_quat (w,x,y,z) = [0.0, 1.0, 0.0, 0.0]


In [20]:
def scatter(mask, name, color, size):
    p = pts[mask]
    return go.Scatter3d(
        x=p[:, 0], y=p[:, 1], z=p[:, 2],
        mode="markers",
        name=f"{name} ({int(mask.sum())})",
        marker=dict(size=size, color=color, opacity=0.85),
    )


def _quat_to_R(quat_wxyz):
    """SAPIEN [w,x,y,z] -> 3x3 rotation matrix (columns = local x,y,z axes in world)."""
    q = np.asarray(quat_wxyz, dtype=np.float64).reshape(4)
    n = np.linalg.norm(q)
    if n < 1e-12:
        return np.eye(3)
    w, x, y, z = (q / n).tolist()
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
        [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
        [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)],
    ])


def frame_triad(origin, quat_wxyz, *, length=0.06, name="frame", width=6):
    """Return 3 Scatter3d line traces for the x(red)/y(green)/z(blue) axes of a pose.

    The three columns of R are the local axis directions expressed in world coords;
    for a downward gripper the local +z (blue) points along the approach axis.
    """
    origin = np.asarray(origin, dtype=np.float64).reshape(3)
    R = _quat_to_R(quat_wxyz)
    axis_colors = [("x", "red"), ("y", "green"), ("z", "blue")]
    traces = []
    for i, (axis_name, color) in enumerate(axis_colors):
        tip = origin + length * R[:, i]
        traces.append(go.Scatter3d(
            x=[origin[0], tip[0]], y=[origin[1], tip[1]], z=[origin[2], tip[2]],
            mode="lines",
            line=dict(color=color, width=width),
            name=f"{name}:{axis_name}",
            showlegend=(i == 0),
            legendgroup=name,
        ))
    return traces


traces = [
    scatter(is_scene, "scene", "lightgray", 2),
    scatter(is_robot, "robot", "royalblue", 3),
    scatter(is_goal, f"goal marker ({goal_marker_style})", "crimson", 4),
]

# TCP + target anchors as distinct diamonds.
traces.append(go.Scatter3d(
    x=[tcp_xyz[0]], y=[tcp_xyz[1]], z=[tcp_xyz[2]], mode="markers",
    name="TCP", marker=dict(size=6, color="green", symbol="diamond"),
))
traces.append(go.Scatter3d(
    x=[target_xyz[0]], y=[target_xyz[1]], z=[target_xyz[2]], mode="markers",
    name="target_position", marker=dict(size=6, color="orange", symbol="x"),
))

# --- TCP triad: where the end-effector lands and how it is oriented ---
# tcp_pose = [x, y, z, qw, qx, qy, qz] (SAPIEN scalar-first quaternion).
tcp_quat = tcp_ep[rel, 3:7]
traces += frame_triad(tcp_xyz, tcp_quat, length=0.06, name="TCP frame", width=7)

# --- Goal-pose triad: the commanded 6D target orientation (if goal_quat exists) ---
if goal_quat_ep is not None:
    traces += frame_triad(
        target_xyz, goal_quat_ep[rel], length=0.06, name="goal frame", width=4
    )

fig = go.Figure(data=traces)
fig.update_layout(
    title=(
        f"{DATASET.name} - episode {EPISODE}, frame {row} (marker: {goal_marker_style})"
        "  |  axes: x=red y=green z=blue (z = approach)"
    ),
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="data",
    ),
    legend=dict(itemsizing="constant"),
    margin=dict(l=0, r=0, t=40, b=0),
    height=750,
)
fig.show()

## Optional: zoom to just the goal-marker triad

Renders only the goal-marker slots (plus the target anchor) so the oriented triad
shape — origin cluster + three unequal arms — is clearly visible and you can verify
the orientation matches `goal_quat`.

In [13]:
gp = pts[is_goal]
if gp.shape[0] == 0:
    print("No goal-marker points at this frame (invalid or goal_marker_points=0).")
else:
    # Color goal points by distance from the target so the arms are distinguishable.
    d = np.linalg.norm(gp - target_xyz[None, :], axis=1)
    figm = go.Figure(data=[
        go.Scatter3d(
            x=gp[:, 0], y=gp[:, 1], z=gp[:, 2], mode="markers",
            marker=dict(size=4, color=d, colorscale="Viridis", colorbar=dict(title="dist")),
            name="goal marker",
        ),
        go.Scatter3d(
            x=[target_xyz[0]], y=[target_xyz[1]], z=[target_xyz[2]], mode="markers",
            name="target", marker=dict(size=6, color="orange", symbol="x"),
        ),
    ])
    figm.update_layout(
        title=f"goal marker ({goal_marker_style}) — episode {EPISODE}, frame {row}",
        scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        height=650, margin=dict(l=0, r=0, t=40, b=0),
    )
    figm.show()